# Загрузка данных

In [ ]:
from pathlib import Path

base_path = Path("SemEval-2020/datasets")

articles_path = base_path / "train-articles"
si_labels_path = base_path / "train-labels-task1-span-identification"
tc_labels_path = base_path / "train-labels-task2-technique-classification"

In [ ]:
articles = {}

for txt in articles_path.glob("*.txt"):
    article_id = txt.stem.replace("article", "")
    articles[article_id] = txt.read_text(encoding="utf-8")

print(f"статей вот столько штук: {len(articles)}")

In [ ]:
si_labels = {}

for label in si_labels_path.glob("*.labels"):
    article_id = label.stem.replace("article", "").replace(".task1-SI", "")
    spans = []

    for line in label.read_text(encoding="utf-8").splitlines():
        _, start, end = line.split("\t")
        spans.append((int(start), int(end)))

    si_labels[article_id] = spans

print(next(iter(si_labels.items())))

# BIO-разметка + базовая BiGRU

In [ ]:
import re
token_pattern = re.compile(r"\w+|[^\w\s]")

def build_dataset(articles, si_labels):
  dataset = []
  for article_id, text in articles.items():
    sentence = []
    spans = si_labels.get(article_id, [])
    spans = sorted(spans, key=lambda x: x[0])

    for match in re.finditer(token_pattern, text): 
      token = match.group()
      start_char, end_char = match.span()
      label = "O" 

      for span_start, span_end in spans:
        if end_char <= span_start: 
          break
        if start_char >= span_end: 
          continue
        
        if span_start <= start_char < span_end: 
          if start_char == span_start: 
            label = "B"
          else:
            label = "I"
          break

      sentence.append({
          "word": token,
          "label": label
      })

    if sentence:
      dataset.append(sentence)

  return dataset

In [ ]:
full_data = build_dataset(articles, si_labels)
print(*full_data[7][:20], sep='\n')

In [ ]:
import torch
import numpy as np
import torch.nn as nn
from typing import List, Tuple, Dict
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

In [ ]:
from sklearn.model_selection import train_test_split
train_data, dev_data = train_test_split(full_data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(dev_data, test_size=0.5, random_state=42)

print(f"train: {len(train_data)}, val: {len(val_data)}, test: {len(test_data)}")

In [ ]:
from collections import defaultdict
class Vocabulary:

    def __init__(self, add_begin=True, add_end=True, min_count=1):
        self.add_begin = add_begin
        self.add_end = add_end
        self.min_count = min_count

    def fit(self, data):
        self.symbols_ = ["<PAD>", "<UNK>", "<BEGIN>", "<END>"]
        symbol_counts = defaultdict(int)
        for text in data:
            for letter in set(text):
                symbol_counts[letter] += 1
        self.symbols_ += [letter for letter, count in symbol_counts.items() if count >= self.min_count]
        self.symbol_codes_ = {letter: index for index, letter in enumerate(self.symbols_)}
        return self

    @property
    def unk(self):
        return self.symbol_codes_["<UNK>"]

    @property
    def begin(self):
        return self.symbol_codes_["<BEGIN>"]

    @property
    def end(self):
        return self.symbol_codes_["<END>"]

    def __call__(self, data):
        if isinstance(data, list) and not isinstance(data[0], str):
            return [self.__call__(text) for text in data]
        indexes = [self.symbol_codes_.get(symbol, self.unk) for symbol in data]
        if self.add_begin:
            indexes = [self.begin] + indexes
        if self.add_end:
            indexes = indexes + [self.end]
        return indexes

In [ ]:
class SequenceDataset(Dataset):

    def __init__(self, data, vocabs=None, fields=None, vocab_params=None,
                 add_begin=False, add_end=False, device="cuda"):
        vocab_params = vocab_params or dict()
        self.add_begin = add_begin
        self.add_end = add_end
        if vocabs is None:
            if fields is None:
                raise ValueError("You should pass `fields` to train `vocabs` if `vocabs` are not available.")
            vocabs = dict()
            for field in fields:
                curr_vocab_params = vocab_params.get(field, dict())
                curr_vocab_params["add_begin"] = add_begin
                curr_vocab_params["add_end"] = add_end
                vocab = Vocabulary(**curr_vocab_params)
                data_for_vocab = [[elem[field] for elem in sent] for sent in data]
                vocabs[field] = vocab.fit(data_for_vocab)
        self.fields = fields
        self.vocabs = vocabs
        self.data = data
        self.device = device

    def _make_mask(self, item):
        answer = [True for _ in item]
        if self.add_begin:
            answer = [False] + answer
        if self.add_end:
            answer.append(False)
        return answer

    def __getitem__(self, index):
        answer = dict()
        for field, vocab in self.vocabs.items():
            answer_field = self.fields.get(field, field)
            answer[answer_field] = vocab([elem[field] for elem in self.data[index]])
        answer["mask"] = self._make_mask(self.data[index])
        answer = {key: torch.tensor(value, dtype=torch.int64).to(self.device) for key, value in answer.items()}
        answer["index"] = index
        return answer

    def __len__(self):
        return len(self.data)

In [ ]:
fields={"word": "input_ids", "label": "labels"}

X_train = SequenceDataset(train_data, fields=fields, vocab_params={"word": {"min_count": 3}, "label": {"min_count": 1}}, add_begin=True, add_end=True, device="cuda")
vocabs = X_train.vocabs
X_val = SequenceDataset(val_data, fields=fields, vocabs=vocabs, add_begin=True, add_end=True, device="cuda")
X_test = SequenceDataset(test_data, fields=fields, vocabs=vocabs, add_begin=True, add_end=True, device="cuda")

In [ ]:
def collate_fn(samples, dtype=torch.int64, keys=None):
    if keys is None:
        keys = ["input_ids", "labels", "mask"]
    device = samples[0]["input_ids"].device
    lengths = [elem["input_ids"].shape[0] for elem in samples]
    L = max(elem["input_ids"].shape[0] for elem in samples)

    answer = dict()
    for key in keys:
        answer[key] = torch.stack([
            torch.cat([
                elem[key],
                torch.zeros(size=(L-len(elem[key]),), dtype=dtype).to(device)
            ]) for elem in samples
        ])
    answer["index"] = np.array([elem["index"] for elem in samples])
    return answer

In [ ]:
class BasicNeuralTagger(nn.Module):

    def __init__(self, vocab_size, labels_number, lr=0.0001,
                 device="cpu", **kwargs):
        super(BasicNeuralTagger, self).__init__()
        self.vocab_size = vocab_size
        self.labels_number = labels_number
        self.build_network(vocab_size, labels_number, **kwargs)
        self.criterion = nn.NLLLoss(ignore_index=-100, reduction="mean")
        self.device = device
        if self.device is not None:
            self.to(self.device)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)

    def build_network(self, **kwargs):
        raise NotImplementedError("You should implement network construction in your derived class.")

    def forward(self, inputs):
        raise NotImplementedError("You should implement forward pass in your derived class.")

    def train_on_batch(self, input_ids, labels, mask=None, **kwargs):
        self.train()
        self.optimizer.zero_grad()
        batch_output = self._validate(input_ids, labels, mask=mask, **kwargs)
        batch_output["loss"].backward()
        self.optimizer.step()
        return batch_output

    def validate_on_batch(self, input_ids, labels, mask=None, **kwargs):
        self.eval()
        with torch.no_grad():
            return self._validate(input_ids, labels, mask=mask, **kwargs)

    def _validate(self, input_ids, labels, mask=None, **kwargs):
        if self.device is not None:
            input_ids, labels = input_ids.to(self.device), labels.to(self.device)
            if mask is not None:
                mask = mask.to(self.device)
        batch_output = self(input_ids, **kwargs)
        if mask is not None:
            labels = torch.where(mask.bool(), labels, -100)
        loss = self.criterion(batch_output["log_probs"].permute(0, 2, 1), labels)
        batch_output["loss"] = loss
        return batch_output

In [ ]:
class MultilayerRNNTagger(BasicNeuralTagger):

    def build_network(self, vocab_size, labels_number, embeddings_dim=32,
                      n_layers=1, n_hidden=128, dropout=0.0):
        self.n_layers = n_layers
        self.n_hidden = n_hidden
        self.embedding = nn.Embedding(vocab_size, embeddings_dim, padding_idx=0)
        self.rnn = torch.nn.GRU(embeddings_dim, self.n_hidden, self.n_layers,
                                batch_first=True, bidirectional=True, dropout=dropout)
        self.dropout = torch.nn.Dropout(dropout)
        self.dense = nn.Linear(2*self.n_hidden, labels_number)
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, input_ids, **kwargs):
        if self.device is not None:
            input_ids = input_ids.to(self.device)
        embeddings = self.embedding(input_ids)
        rnn_outputs, rnn_state = self.rnn(embeddings)
        rnn_outputs = self.dropout(rnn_outputs)
        logits = self.dense(rnn_outputs)
        log_probs = self.log_softmax(logits)
        _, labels = torch.max(log_probs, dim=-1)
        return {"log_probs": log_probs, "labels": labels}

In [ ]:
def merge_spans(spans):
  if not spans:
    return []
  spans = sorted(spans, key=lambda x: x[0])
  merged = [spans[0]]
  for current in spans[1:]:
      prev_start, prev_end = merged[-1]
      curr_start, curr_end = current
      if curr_start <= prev_end:
          merged[-1] = (prev_start, max(prev_end, curr_end))
      else:
          merged.append(current)
  return merged

In [ ]:
def compute_dataset_scores(all_pred, all_gold):
    #Реализация по формуле из SemEval 2020 Task 11
    S = sum(len(merge_spans(p)) for p in all_pred)
    T = sum(len(merge_spans(g)) for g in all_gold)
    if S == 0:
        precision = 0
    else:
        precision_sum = 0
        for pred_spans, gold_spans in zip(all_pred, all_gold):
            pred_spans = merge_spans(pred_spans)
            gold_spans = merge_spans(gold_spans)

            for s in pred_spans:
                for t in gold_spans:
                    overlap = max(0, min(s[1], t[1]) - max(s[0], t[0]))
                    if t[1] > t[0]:
                        precision_sum += overlap / (t[1] - t[0])

        precision = precision_sum / S
    if T == 0:
        recall = 0
    else:
        recall_sum = 0
        for pred_spans, gold_spans in zip(all_pred, all_gold):
            pred_spans = merge_spans(pred_spans)
            gold_spans = merge_spans(gold_spans)

            for s in pred_spans:
                for t in gold_spans:
                    overlap = max(0, min(s[1], t[1]) - max(s[0], t[0]))
                    if s[1] > s[0]:
                        recall_sum += overlap / (s[1] - s[0])
        recall = recall_sum / T
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

In [ ]:
def bio_to_spans(labels_ids, mask, id2label):
  spans = []
  start = None
  for i, (lab_id, m) in enumerate(zip(labels_ids, mask)):
    if not m:
      continue
    label = id2label[lab_id]
    if label == "B":
      if start is not None:
        spans.append((start, i))
      start = i
    elif label == "I":
      if start is None:
        start = i
    else:
      if start is not None:
        spans.append((start, i))
        start = None
  if start is not None:
    spans.append((start, len(labels_ids)))
  return spans

In [ ]:
def update_metrics(metrics, batch_output, batch_labels, mask=None, to_print=False):
    n_batches = metrics["n_batches"]
    metrics["loss"] = (metrics["loss"] * n_batches + batch_output["loss"].item()) / (n_batches + 1)
    metrics["n_batches"] += 1
    if mask is not None:
        mask = mask.cpu().numpy().astype("int")
    else:
        mask = (batch_labels != 0).cpu().numpy().astype("int")
    are_equal = (batch_output["labels"] == batch_labels).cpu().numpy().astype("int")
    curr_correct = (are_equal * mask).sum()
    curr_total = mask.sum()
    metrics["correct"] += (are_equal * mask).sum()
    metrics["total"] += mask.sum()
    are_seq_correct = np.min(np.maximum(are_equal, 1-mask), axis=1)
    metrics["sent_correct"] += are_seq_correct.sum()
    metrics["sent_total"] += mask.shape[0]
    metrics["accuracy"] = metrics["correct"] / max(metrics["total"], 1)
    metrics["sent_accuracy"] = metrics["sent_correct"] / max(metrics["sent_total"], 1)

def do_epoch(model, dataloader, mode="validate", epoch=1):
    metrics = {
        "correct": 0, "total": 0,
        "sent_correct": 0, "sent_total": 0,
        "loss": 0.0, "n_batches": 0
    }

    all_pred_spans = []
    all_gold_spans = []

    id2label = {v: k for k, v in dataloader.dataset.vocabs["label"].symbol_codes_.items()}

    func = model.train_on_batch if mode == "train" else model.validate_on_batch
    progress_bar = tqdm(dataloader, leave=True)
    progress_bar.set_description(f"{mode}, epoch={epoch}")

    for batch in progress_bar:
        batch_output = func(**batch)

        update_metrics(metrics, batch_output, batch["labels"], mask=batch["mask"])

        preds = batch_output["labels"].detach().cpu().numpy()
        golds = batch["labels"].detach().cpu().numpy()
        masks = batch["mask"].detach().cpu().numpy()

        for p, g, m in zip(preds, golds, masks):
            pred_spans = bio_to_spans(p, m, id2label)
            gold_spans = bio_to_spans(g, m, id2label)

            all_pred_spans.append(pred_spans)
            all_gold_spans.append(gold_spans)

        progress_bar.set_postfix({
            "loss": round(metrics["loss"], 4),
            "acc": round(100 * metrics["accuracy"], 2),
            "sent_acc": round(100 * metrics["sent_accuracy"], 2)
        })

    precision, recall, f1 = compute_dataset_scores(all_pred_spans, all_gold_spans)

    metrics["precision"] = precision
    metrics["recall"] = recall
    metrics["f1"] = f1

    print(f"\nP: {precision:.4f} | R: {recall:.4f} | F1: {f1:.4f}")
    return metrics

In [ ]:
train_dataloader = DataLoader(X_train, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(X_val, batch_size=16, shuffle=False, collate_fn=collate_fn)
test_dataloader = DataLoader(X_test, batch_size=16, shuffle=False, collate_fn=collate_fn)
NEPOCHS = 7

model = MultilayerRNNTagger(vocab_size=len(vocabs["word"].symbols_),
                            labels_number=len(vocabs["label"].symbols_),
                            embeddings_dim=300,
                            n_layers=2, dropout=0.1,
                            n_hidden=192,
                            device="cuda")

for epoch in range(NEPOCHS):
    do_epoch(model, train_dataloader, mode="train", epoch=epoch+1)
    epoch_metrics = do_epoch(model, val_dataloader, mode="validate", epoch=epoch+1)
do_epoch(model, val_dataloader, mode="validate", epoch="evaluate")
do_epoch(model, test_dataloader, mode="validate", epoch="evaluate")